In [0]:
customers_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_customers")
exchange_rates_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_exchange_rates")
invoice_line_items_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_invoice_line_items")
invoices_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_invoices")
payments_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_payments")
products_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_products")
regions_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_regions")

In [0]:
display(customers_df)

In [0]:
display(payments_df)

In [0]:
display(products_df)


In [0]:
display(invoice_line_items_df)

In [0]:
display(invoices_df)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
fact_invoice_line_items_df = invoice_line_items_df.drop("product_name", "customer_name", "region","original_currency")
fact_payments_df = payments_df.drop("payment_method", "customer_name", "region","original_currency","payment_date")

display(fact_invoice_line_items_df)
display(fact_payments_df)

dim_customers_df = customers_df
dim_products_df = products_df
dim_regions_df = regions_df
dim_invoices_df = invoices_df  
fact_invoice_line_items_df = fact_invoice_line_items_df.withColumn(
    'usd_revenue',
    F.round(col('quantity') * (1 - col("discount")) * col('unit_price_usd'),2)
)

In [0]:
dim_products_df = dim_products_df.drop("cost_price", "original_currency","original_currency","cost_price_usd")
display(dim_products_df)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "curated_wholesale_distributor"
table_name = "fact_payments"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

fact_payments_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "curated_wholesale_distributor"
table_name = "fact_invoice_line_items"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

fact_invoice_line_items_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "curated_wholesale_distributor"
table_name = "dim_customers"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

dim_customers_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "curated_wholesale_distributor"
table_name = "dim_products"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

dim_products_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "curated_wholesale_distributor"
table_name = "dim_invoices"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

dim_invoices_df.write.mode("overwrite").saveAsTable(table_path)